<a href="https://colab.research.google.com/github/zogydinesh/ITA-0616/blob/main/irisprog4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd

# Dataset: Iris - UCI Machine Learning Repository

# ---- 1. Load CSV ----
df = pd.read_csv('/content/irisprog4.csv')
X = df.iloc[:, :-1].values.astype(float)
classes = sorted(df['species'].unique())
y = np.array([classes.index(s) for s in df['species']])

# ---- 2. Normalize inputs (0 to 1) and one-hot encode labels ----
X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
Y = np.eye(len(classes))[y]

# ---- 3. Train / test split (80% / 20%) ----
np.random.seed(42)
idx = np.random.permutation(len(X))
split = int(0.8 * len(X))
Xtr, Xte = X[idx[:split]], X[idx[split:]]
Ytr, Yte = Y[idx[:split]], Y[idx[split:]]
print("Train size:", len(Xtr), " Test size:", len(Xte), "\n")

# ---- 4. Network: 4 inputs -> 6 hidden -> 3 outputs ----
n_in, n_hid, n_out = X.shape[1], 6, len(classes)
W1 = np.random.uniform(-1, 1, (n_in, n_hid)); b1 = np.zeros((1, n_hid))
W2 = np.random.uniform(-1, 1, (n_hid, n_out)); b2 = np.zeros((1, n_out))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

lr, epochs = 0.5, 3000

# ---- 5. Training with Backpropagation ----
for epoch in range(1, epochs + 1):
    # Forward pass
    h = sigmoid(Xtr @ W1 + b1)
    out = sigmoid(h @ W2 + b2)

    # Error (mean squared error)
    err = Ytr - out
    loss = np.mean(err ** 2)

    # Backward pass (delta rule)
    d_out = err * out * (1 - out)               # output layer delta
    d_hid = (d_out @ W2.T) * h * (1 - h)        # hidden layer delta

    # Update weights
    W2 += lr * h.T @ d_out / len(Xtr)
    b2 += lr * d_out.mean(axis=0, keepdims=True)
    W1 += lr * Xtr.T @ d_hid / len(Xtr)
    b1 += lr * d_hid.mean(axis=0, keepdims=True)

    if epoch % 500 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d}  Loss = {loss:.5f}")

# ---- 6. Testing ----
def predict(Xin):
    return np.argmax(sigmoid(sigmoid(Xin @ W1 + b1) @ W2 + b2), axis=1)

train_acc = np.mean(predict(Xtr) == np.argmax(Ytr, axis=1)) * 100
test_acc = np.mean(predict(Xte) == np.argmax(Yte, axis=1)) * 100
print(f"\nTraining accuracy: {train_acc:.2f}%")
print(f"Test accuracy:     {test_acc:.2f}%")

print("\nSample test predictions:")
for i in range(5):
    print(f"  Actual: {classes[np.argmax(Yte[i])]:<12} Predicted: {classes[predict(Xte[i:i+1])[0]]}")

Train size: 120  Test size: 30 

Epoch    1  Loss = 0.29416
Epoch  500  Loss = 0.10642
Epoch 1000  Loss = 0.08436
Epoch 1500  Loss = 0.06270
Epoch 2000  Loss = 0.04471
Epoch 2500  Loss = 0.03372
Epoch 3000  Loss = 0.02733

Training accuracy: 96.67%
Test accuracy:     96.67%

Sample test predictions:
  Actual: versicolor   Predicted: versicolor
  Actual: setosa       Predicted: setosa
  Actual: versicolor   Predicted: versicolor
  Actual: versicolor   Predicted: versicolor
  Actual: setosa       Predicted: setosa
